# 系列 2：基于 Qdrant 的本地开源 RAG

本笔记本验证文章的本地开源 RAG 路径。它使用 Qdrant 内存模式、本地示例文档、CPU 友好的 FastEmbed 嵌入模型，以及透明的本地答案合成器。

目标是首先使 RAG 工作流程可见：加载文档、分块、创建真实的本地嵌入、存储向量、检索证据、重新排序并返回来源，然后可选择使用 Ollama 进行本地 LLM 生成。


## 安装依赖

从仓库根目录开始：

```bash
python -m venv .venv-series2
.venv-series2\Scripts\activate
python -m pip install -r requirements/open-source-rag.txt
```


In [ ]:
import importlib.metadata
import re
import sys
from pathlib import Path

from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models

packages = ["qdrant-client", "fastembed", "python-dotenv", "nbclient", "nbformat", "ipykernel", "numpy"]
versions = {name: importlib.metadata.version(name) for name in packages}
print("Python:", sys.version.split()[0])
for name, version in versions.items():
    print(f"{name}: {version}")

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "sample_data").exists():
    repo_root = Path.cwd().parent

sample_dir = repo_root / "sample_data"
sample_files = ["course_ai_guidance.md", "school_ai_policy.md"]

documents = []
for file_name in sample_files:
    path = sample_dir / file_name
    documents.append({"source": path.name, "text": path.read_text(encoding="utf-8")})

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print("-", doc["source"])


In [ ]:
def chunk_markdown(document):
    title = None
    current_heading = None
    current_lines = []
    chunks = []

    def flush():
        if current_heading and current_lines:
            content = "\n".join(line for line in current_lines).strip()
            if content:
                chunks.append({
                    "id": f"{document['source']}::{len(chunks)}",
                    "source": document["source"],
                    "title": title or document["source"],
                    "sectionHeading": current_heading,
                    "content": content,
                    "documentVersion": "local-sample-v1",
                    "permissions": ["students", "instructors"],
                })

    for raw_line in document["text"].splitlines():
        line = raw_line.strip()
        if line.startswith("# "):
            title = line[2:].strip()
        elif line.startswith("## "):
            flush()
            current_heading = line[3:].strip()
            current_lines = []
        elif line:
            current_lines.append(line)

    flush()
    return chunks

chunks = []
for document in documents:
    chunks.extend(chunk_markdown(document))

print(f"Created {len(chunks)} chunks")
for chunk in chunks:
    print(f"- {chunk['source']} / {chunk['sectionHeading']}")

In [ ]:
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
embedding_model = TextEmbedding(model_name=EMBEDDING_MODEL_NAME)

def tokenize(text):
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    expanded = []
    for token in tokens:
        expanded.append(token)
        if token.endswith("s") and len(token) > 3:
            expanded.append(token[:-1])
    return expanded

texts_to_embed = [
    f"{chunk['title']} {chunk['sectionHeading']} {chunk['content']}"
    for chunk in chunks
]
chunk_vectors = list(embedding_model.embed(texts_to_embed))
VECTOR_SIZE = len(chunk_vectors[0])

for chunk, vector in zip(chunks, chunk_vectors):
    chunk["vector"] = vector

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Generated {len(chunks)} local embeddings with size {VECTOR_SIZE}")

In [ ]:
collection_name = "school_policy_local"
client = QdrantClient(":memory:")

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE),
)

points = []
for idx, chunk in enumerate(chunks):
    payload = {key: chunk[key] for key in ["source", "title", "sectionHeading", "content", "documentVersion", "permissions"]}
    points.append(models.PointStruct(id=idx, vector=chunk["vector"].tolist(), payload=payload))

client.upsert(collection_name=collection_name, points=points)
count = client.count(collection_name=collection_name, exact=True).count
print(f"Qdrant collection initialized: {collection_name}")
print(f"Vectors inserted: {count}")

In [ ]:
question = "Can I use generative AI for my final assignment?"
query_vector = list(embedding_model.embed([question]))[0].tolist()

raw_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True,
).points

query_terms = set(tokenize(question))

def rerank_score(result):
    payload = result.payload
    heading_terms = set(tokenize(payload["sectionHeading"]))
    content_terms = set(tokenize(payload["content"]))
    heading_overlap = len(query_terms & heading_terms)
    content_overlap = len(query_terms & content_terms)
    return result.score + (0.12 * heading_overlap) + (0.02 * content_overlap)

results = sorted(raw_results, key=rerank_score, reverse=True)[:3]

print("Question:", question)
print("Top retrieved chunks after lightweight reranking:")
for rank, result in enumerate(results, start=1):
    payload = result.payload
    print(f"{rank}. vector_score={result.score:.4f} rerank_score={rerank_score(result):.4f} source={payload['source']} section={payload['sectionHeading']}")
    print("   ", payload["content"][:180].replace("\n", " "))

In [ ]:
top = results[0].payload
answer = (
    "Based on the retrieved policy section, students may use generative AI for "
    "brainstorming, outlining, grammar feedback, and code explanation when the "
    "instructor allows it. They should not submit AI-generated work as their own, "
    "and they should include a disclosure when AI tools are used."
)

print("Answer:")
print(answer)
print("\nSource:")
print(f"{top['source']} / {top['sectionHeading']}")

## 使用 Ollama 和 Phi-4-mini 本地生成答案

上面默认的答案组合器是确定性的，因此笔记本可以在任何地方运行。要启用本地 LLM 生成，请在仓库根目录创建 `.env` 文件并设置：

```text
SERIES2_OLLAMA_BASE_URL=http://localhost:11434
SERIES2_OLLAMA_MODEL=phi4-mini:3.8b
```

本部分重用来自 Qdrant 的相同检索证据并将其发送到 Ollama。如果 `.env` 中不包含 `SERIES2_OLLAMA_MODEL`，则跳过生成单元。


In [ ]:
import json
import os
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv(repo_root / ".env")

def build_evidence(retrieved_results):
    evidence_blocks = []
    for idx, result in enumerate(retrieved_results, start=1):
        payload = result.payload
        evidence_blocks.append(
            f"[{idx}] Source: {payload['source']} / {payload['sectionHeading']}\n"
            f"{payload['content']}"
        )
    return "\n\n".join(evidence_blocks)

evidence = build_evidence(results)
answer_prompt = (
    "Answer the question using only the evidence below. "
    "If the evidence is insufficient, say that the provided documents do not contain enough information. "
    "End with a Sources line that lists the source file and section.\n\n"
    f"Question: {question}\n\nEvidence:\n{evidence}"
)

print("Prepared evidence for local Ollama generation")
print(evidence[:500])

In [ ]:
ollama_generation_status = "skipped_missing_environment"
ollama_answer = None
ollama_model = os.getenv("SERIES2_OLLAMA_MODEL")
ollama_base_url = os.getenv("SERIES2_OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")

if not ollama_model:
    print("Ollama generation skipped. Set SERIES2_OLLAMA_MODEL in .env to enable it.")
else:
    request_body = {
        "model": ollama_model,
        "messages": [
            {"role": "system", "content": "You answer only from retrieved evidence and cite sources."},
            {"role": "user", "content": answer_prompt},
        ],
        "stream": False,
    }
    request = urllib.request.Request(
        f"{ollama_base_url}/api/chat",
        data=json.dumps(request_body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        ollama_generation_status = "running_ollama"
        with urllib.request.urlopen(request, timeout=120) as response:
            response_payload = json.loads(response.read().decode("utf-8"))
        ollama_answer = response_payload.get("message", {}).get("content", "")
        ollama_generation_status = "completed_ollama"
    except (urllib.error.URLError, TimeoutError) as exc:
        ollama_generation_status = "skipped_ollama_unreachable"
        print(f"Ollama generation skipped because the local server was not reachable: {exc}")

print("Ollama generation status:", ollama_generation_status)
if ollama_answer:
    print(ollama_answer)

In [ ]:
verification = {
    "documents_loaded": len(documents),
    "chunks_created": len(chunks),
    "collection_name": collection_name,
    "vectors_inserted": count,
    "retrieval_question": question,
    "top_source": results[0].payload["source"],
    "top_section": results[0].payload["sectionHeading"],
    "embedding_path": f"local FastEmbed model: {EMBEDDING_MODEL_NAME}",
    "generation_path": "local transparent answer composer",
    "ollama_generation_status": ollama_generation_status,
}

print("Verification summary")
for key, value in verification.items():
    print(f"{key}: {value}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
